In [11]:
import sqlite3
import pandas as pd
import json
import os
from dotenv import load_dotenv

load_dotenv()
DB_NAME = os.getenv('DB_NAME', 'jobscope.db')
conn = sqlite3.connect(DB_NAME)

In [13]:
# Load both tables into DataFrames
raw_df = pd.read_sql('SELECT * FROM raw_jobs', conn)
clean_df = pd.read_sql('SELECT * FROM clean_jobs', conn)

print(f'Raw jobs: {len(raw_df)}')
print(f'Clean jobs: {len(clean_df)}')

Raw jobs: 2765
Clean jobs: 2368


In [14]:
raw_df.head()

,id,source,external_id,title,company,description,location,salary_min,salary_max,salary_is_predicted,category,url,date_posted,date_collected,search_term
0,1,adzuna,5636282281,Junior Data Scientist,Newto Training,Are you ready to start a new career in Data An...,UK,44849.22,44849.22,1,IT Jobs,https://www.adzuna.co.uk/jobs/land/ad/56362822...,2026-02-19T18:28:35Z,2026-03-17T12:09:47.429516,data scientist
1,2,adzuna,5647306272,Data Scientist,Provide,Data Scientist – MRO AI Solutions (Embedded in...,"Heathrow, Hounslow",70672.75,70672.75,1,IT Jobs,https://www.adzuna.co.uk/jobs/land/ad/56473062...,2026-02-27T14:55:45Z,2026-03-17T12:09:47.429516,data scientist
2,3,adzuna,5647297574,Data Scientist,True North Group,Data Scientist Newcastle upon Tyne (Hybrid Wor...,"Newcastle Upon Tyne, Tyne & Wear",41343.96,41343.96,1,IT Jobs,https://www.adzuna.co.uk/jobs/land/ad/56472975...,2026-02-27T14:30:03Z,2026-03-17T12:09:47.429516,data scientist
3,4,adzuna,5642964323,Senior Data Scientist,E-Solutions IT Services UK Ltd,Role: Senior Data Scientist Location: Manchest...,"Manchester, Greater Manchester",51727.38,51727.38,1,Accounting & Finance Jobs,https://www.adzuna.co.uk/jobs/land/ad/56429643...,2026-02-23T23:29:21Z,2026-03-17T12:09:47.429516,data scientist
4,5,adzuna,5645753150,Lead Data Scientist,Ocho,Job description Lead Data Scientist Location: ...,"Belfast, Northern Ireland",80000.00,80000.00,1,IT Jobs,https://www.adzuna.co.uk/jobs/land/ad/56457531...,2026-02-26T03:27:24Z,2026-03-17T12:09:47.429516,data scientist


In [15]:
# Quick look at clean data
clean_df.head()

,id,raw_job_id,title_original,title_normalized,role_category,company,description_clean,location_raw,location_city,location_region,salary_min,salary_max,salary_mid,has_real_salary,extracted_skills,date_posted,seniority,job_source
0,6781,1,Junior Data Scientist,Junior Data Scientist,Data Scientist,Newto Training,are you ready to start a new career in data an...,UK,UK-wide,UK-wide,44849.22,44849.22,44849.22,0,"[""data analysis"", ""data analytics"", ""excel""]",2026-02-19T18:28:35Z,junior,adzuna
1,6782,2,Data Scientist,Data Scientist,Data Scientist,Provide,data scientist – mro ai solutions (embedded in...,"Heathrow, Hounslow",Heathrow,Other UK,70672.75,70672.75,70672.75,0,[],2026-02-27T14:55:45Z,mid,adzuna
2,6783,3,Data Scientist,Data Scientist,Data Scientist,True North Group,data scientist newcastle upon tyne (hybrid wor...,"Newcastle Upon Tyne, Tyne & Wear",Newcastle Upon Tyne,North East,41343.96,41343.96,41343.96,0,"[""machine learning""]",2026-02-27T14:30:03Z,mid,adzuna
3,6784,4,Senior Data Scientist,Senior Data Scientist,Data Scientist,E-Solutions IT Services UK Ltd,role: senior data scientist location: manchest...,"Manchester, Greater Manchester",Manchester,North West,51727.38,51727.38,51727.38,0,[],2026-02-23T23:29:21Z,senior,adzuna
4,6785,5,Lead Data Scientist,Lead Data Scientist,Data Scientist,Ocho,job description lead data scientist location: ...,"Belfast, Northern Ireland",Belfast,Northern Ireland,80000.00,80000.00,80000.00,0,"[""leadership"", ""mentoring""]",2026-02-26T03:27:24Z,senior,adzuna


In [16]:
# Column types and nulls
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2368 entries, 0 to 2367
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2368 non-null   int64  
 1   raw_job_id         2368 non-null   int64  
 2   title_original     2368 non-null   object 
 3   title_normalized   2368 non-null   object 
 4   role_category      2368 non-null   object 
 5   company            2368 non-null   object 
 6   description_clean  2368 non-null   object 
 7   location_raw       2368 non-null   object 
 8   location_city      2368 non-null   object 
 9   location_region    2368 non-null   object 
 10  salary_min         2225 non-null   float64
 11  salary_max         2225 non-null   float64
 12  salary_mid         2219 non-null   float64
 13  has_real_salary    2368 non-null   int64  
 14  extracted_skills   2368 non-null   object 
 15  date_posted        2368 non-null   object 
 16  seniority          2368 

In [5]:
# Parse the extracted_skills JSON column into a Python list
clean_df['skills_list'] = clean_df['extracted_skills'].apply(json.loads)
clean_df['num_skills'] = clean_df['skills_list'].apply(len)

clean_df[['title_original', 'role_category', 'seniority', 'location_region', 'salary_mid', 'num_skills']].head(10)

,title_original,role_category,seniority,location_region,salary_mid,num_skills
0,Junior Data Scientist,Data Scientist,junior,UK-wide,44849.22,3
1,Data Scientist,Data Scientist,mid,Other UK,70672.75,0
2,Data Scientist,Data Scientist,mid,North East,41343.96,1
3,Senior Data Scientist,Data Scientist,senior,North West,51727.38,0
4,Lead Data Scientist,Data Scientist,senior,Northern Ireland,80000.00,2
5,Data Science Trainee,Data Scientist,junior,East Midlands,35000.00,0
6,Data Science Trainee,Data Scientist,junior,East Midlands,35000.00,0
7,Data Science Trainee,Data Scientist,junior,South West,35000.00,0
8,Data Science Trainee,Data Scientist,junior,South East,35000.00,0
9,Data Science Trainee,Data Scientist,junior,North West,35000.00,0


In [6]:
# Quick value counts
print('--- Role Categories ---')
print(clean_df['role_category'].value_counts())
print('\n--- Seniority ---')
print(clean_df['seniority'].value_counts())
print('\n--- Top 10 Regions ---')
print(clean_df['location_region'].value_counts().head(10))

--- Role Categories ---
AI Engineer       682
Data Analyst      561
Data Scientist    341
Data Engineer     329
ML Engineer       257
BI Analyst        158
LLM Engineer       27
NLP Engineer       13
Name: role_category, dtype: int64

--- Seniority ---
mid       1251
junior     592
senior     525
Name: seniority, dtype: int64

--- Top 10 Regions ---
London             1025
Other UK            277
UK-wide             173
North West          157
Yorkshire           133
West Midlands       124
South East          122
South West           83
East of England      77
Scotland             54
Name: location_region, dtype: int64


In [7]:
# Salary overview (only real salaries)
salary_df = clean_df[clean_df['has_real_salary'] == 1].copy()
print(f'Jobs with real salary: {len(salary_df)}')
salary_df['salary_mid'].describe()

Jobs with real salary: 833


count       833.000000
mean      47073.530786
std       27538.548550
min          15.380000
25%       35000.000000
50%       40000.000000
75%       57500.000000
max      360000.000000
Name: salary_mid, dtype: float64

In [10]:
# Don't forget to close when done
conn.close()

In [8]:
# 1. Skipped jobs — what titles are we missing?
print("=== TOP 30 SKIPPED TITLES ===")
skipped = pd.read_sql("""
    SELECT r.title, r.search_term, COUNT(*) as cnt 
    FROM raw_jobs r
    LEFT JOIN clean_jobs c ON c.raw_job_id = r.id
    WHERE c.id IS NULL
    GROUP BY r.title
    ORDER BY cnt DESC
    LIMIT 30
""", conn)
print(skipped.to_string(index=False))

=== TOP 30 SKIPPED TITLES ===
                                                                   title                   search_term  cnt
                                          Trainee Field Service Engineer     machine learning engineer    9
                                                Senior Software Engineer                  LLM engineer    7
                                                     Head of Engineering                  LLM engineer    7
                                   Technical Lead - TypeScript / Node.js                  LLM engineer    6
                                                       Software Engineer                  LLM engineer    5
                                                             IT Engineer     machine learning engineer    5
                                                Finance Business Partner business intelligence analyst    5
                                                   Data Cabling Engineer                 data engineer    

In [4]:
# 2. "Other UK" locations — what are the actual values?
print("\n=== TOP 30 'OTHER UK' LOCATIONS ===")
other_locs = pd.read_sql("""
    SELECT location_raw, COUNT(*) as cnt
    FROM clean_jobs 
    WHERE location_region = 'Other UK'
    GROUP BY location_raw
    ORDER BY cnt DESC
    LIMIT 30
""", conn)
print(other_locs.to_string(index=False))


=== TOP 30 'OTHER UK' LOCATIONS ===
                    location_raw  cnt
                              UK  156
                  United Kingdom   13
Newcastle Upon Tyne, Tyne & Wear   11
                       Sheffield    9
     Cheltenham, Gloucestershire    8
             Telford, Shropshire    7
     Nottingham, Nottinghamshire    7
             Newcastle Upon Tyne    6
                   Milton Keynes    6
        Stevenage, Hertfordshire    5
                          SW21RW    5
                          PR12RL    5
  Milton Keynes, Buckinghamshire    5
                          LU12BQ    5
            Warrington, Cheshire    4
                           W52BJ    4
          Southampton, Hampshire    4
                         SO147LY    4
                          PL13BJ    4
                           N12UD    4
                          MK93HQ    4
                       Leicester    4
     Gloucester, Gloucestershire    4
                   Exeter, Devon    4
             

In [5]:
# 3. Skill extraction check — sample a description that should have Python
print("\n=== SAMPLE DESCRIPTION (first Data Scientist job) ===")
sample = pd.read_sql("""
    SELECT description_clean, extracted_skills
    FROM clean_jobs 
    WHERE role_category = 'Data Scientist'
    LIMIT 1
""", conn)
print(sample['description_clean'].iloc[0][:500])
print("\nExtracted:", sample['extracted_skills'].iloc[0])



=== SAMPLE DESCRIPTION (first Data Scientist job) ===
are you ready to start a new career in data analysis? the demand for data analysts has grown by 20% annually, with experienced professionals earning salaries upwards of £58,000. in today’s digital world, data is critical to business decision-making, making the role of a data analyst indispensable. as skills shortages continue to grow, the demand for qualified entry-level professionals is on the rise. with our data analytics career programme we will provide you with: 8 training modules: excel, s…

Extracted: ["excel"]


In [6]:
# Run this to check if Reed descriptions are longer
import sqlite3, os
from dotenv import load_dotenv
load_dotenv()
conn = sqlite3.connect(os.getenv('DB_NAME', 'jobscope.db'))

for source in ['adzuna', 'reed']:
    rows = conn.execute(
        "SELECT AVG(LENGTH(description)), MAX(LENGTH(description)), MIN(LENGTH(description)) FROM raw_jobs WHERE source = ?",
        (source,)
    ).fetchone()
    print(f"{source}: avg={rows[0]:.0f}, max={rows[1]}, min={rows[2]}")

conn.close()

adzuna: avg=499, max=500, min=105
reed: avg=453, max=453, min=255


In [7]:
import sqlite3, os, pandas as pd
from dotenv import load_dotenv
load_dotenv()
conn = sqlite3.connect(os.getenv('DB_NAME', 'jobscope.db'))

# 1. What's still being skipped?
print("=== TOP 20 STILL-SKIPPED TITLES ===")
skipped = pd.read_sql("""
    SELECT r.title, r.search_term, COUNT(*) as cnt 
    FROM raw_jobs r
    LEFT JOIN clean_jobs c ON c.raw_job_id = r.id
    WHERE c.id IS NULL
    GROUP BY r.title
    ORDER BY cnt DESC
    LIMIT 20
""", conn)
print(skipped.to_string(index=False))



=== TOP 20 STILL-SKIPPED TITLES ===
                                             title                   search_term  cnt
                    Trainee Field Service Engineer     machine learning engineer    9
                                   Finance Analyst                  data analyst    9
                          Senior Software Engineer                  LLM engineer    7
                               Head of Engineering                  LLM engineer    7
             Technical Lead - TypeScript / Node.js                  LLM engineer    6
                                 Software Engineer                  LLM engineer    5
                                       IT Engineer     machine learning engineer    5
                          Finance Business Partner business intelligence analyst    5
                             Data Cabling Engineer                 data engineer    4
                    Technical Director - CIO Level                  LLM engineer    3
                  

In [8]:
# 2. What's still Other UK?
print("\n=== TOP 20 STILL 'OTHER UK' LOCATIONS ===")
other = pd.read_sql("""
    SELECT location_raw, COUNT(*) as cnt
    FROM clean_jobs 
    WHERE location_region = 'Other UK'
    GROUP BY location_raw
    ORDER BY cnt DESC
    LIMIT 20
""", conn)
print(other.to_string(index=False))

conn.close()


=== TOP 20 STILL 'OTHER UK' LOCATIONS ===
                 location_raw  cnt
          Luton, Bedfordshire    4
                  Basingstoke    4
                Wolverhampton    3
                          USA    3
                       TW33EB    3
           Swindon, Wiltshire    3
Stoke-On-Trent, Staffordshire    3
                       RM14GR    3
                   Maidenhead    3
                       IG11DD    3
                       HA90AD    3
                       HA12XY    3
                   Gillingham    3
                       YO19QL    2
        Worthing, West Sussex    2
                     Whiteley    2
                 Warwickshire    2
        Warminster, Wiltshire    2
                       UB81UW    2
                       UB81JG    2
